# PS-C1 — Tunable H₂ VQE across Processor A/B

Team Quantum Mind · Lab 5-based reduced two-qubit VQE. This notebook reports routed circuit costs on the fixed Phase 1 maps and compares measured energies with exact diagonalization.

Edit `config.json` to tune shots, optimizer budget, initial layout, and noise mode. The supplied processor configuration document marks noise rates unverified, so noisy execution remains off until official rates are confirmed.

In [ ]:
# If needed, install the pinned-compatible dependencies in requirements.txt
%pip install -r requirements.txt

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from src.problem import hamiltonian, exact_ground_energy, EXACT_TOTAL_HARTREE
from src.processors import load_processor
from src.visualization import draw_topology
from src.experiment import run

config = json.loads(Path('config.json').read_text())
print('R (Angstrom):', config['geometry_angstrom'])
print('Exact reduced-Hamiltonian ground energy (Hartree):', exact_ground_energy())
print('Challenge reference (Hartree):', EXACT_TOTAL_HARTREE)

## Fixed processor architectures
The maps below come from the supplied simulation configuration PDF. Processor B uses all nine qubits.

In [ ]:
fig_a = draw_topology(load_processor('processor_A'))
fig_b = draw_topology(load_processor('processor_B'))
plt.show()

## Run the A/B experiment
Each energy evaluation uses the Lab 5 ansatz and finite-shot Pauli measurements. The Pauli measurement circuits are separately transpiled against each physical coupling map with the configured seed and optimization level. Results are saved under `results/`.

In [ ]:
table, histories = run()
display(table)

In [ ]:
for name, history in histories.items():
    plt.plot([row[1] for row in history], marker='.', label=name)
plt.axhline(exact_ground_energy(), color='black', linestyle='--', label='exact diagonalization')
plt.xlabel('Objective evaluation'); plt.ylabel('Energy (Hartree)')
plt.title('Shot-based COBYLA VQE convergence'); plt.legend(); plt.grid(alpha=.2); plt.show()

## Interpretation
Compare energy error, transpiled depth, CX count, and SWAP count in the exported table. Because the logical ansatz uses only two qubits, routing overhead may depend strongly on the chosen `initial_layout`; tune it and rerun. The compact Lab 5 Hamiltonian coefficients are for 0.735 Å (near the challenge's 0.74 Å equilibrium). Use Qiskit Nature/PySCF to regenerate integrals if exact 0.740 Å geometry is required.

Finite-shot estimates can fall below the exact eigenvalue due to sampling variance; the underlying expectation still obeys the variational principle.